# Pilot Dataset Analysis

Select and freeze a small metadata-only pilot. No image staging or retrieval.

## Setup

In [1]:
import json
import importlib.util
from pathlib import Path
from collections import Counter

import pandas as pd

working_directory = Path.cwd()
project_root = working_directory
if working_directory.name == "notebooks":
    project_root = working_directory.parent

seed = 42
query_target = 200
queries_per_group = 25
evidence_target = 5_000
news_directory = project_root / "data" / "raw" / "newsclippings"
metadata_directory = project_root / "data" / "raw" / "visualnews_metadata"
manifest_directory = project_root / "data" / "manifests"

## Load Data

In [2]:
with (news_directory / "train.json").open("r", encoding="utf-8") as handle:
    news_object = json.load(handle)
train_annotations = pd.DataFrame(news_object["annotations"])

metadata_splits = {}
for split in ("train", "val", "test"):
    metadata_path = metadata_directory / f"{split}.json"
    with metadata_path.open("r", encoding="utf-8") as handle:
        metadata_splits[split] = json.load(handle)

visualnews_by_id = {}
metadata_split_by_id = {}
for split, metadata_records in metadata_splits.items():
    for key, record in metadata_records.items():
        metadata_id = record["id"]
        if type(metadata_id) is not int or str(metadata_id) != key:
            raise ValueError(f"{split}: invalid metadata ID/key {key!r}")
        if metadata_id in visualnews_by_id:
            raise ValueError(f"Ambiguous metadata ID: {metadata_id}")
        visualnews_by_id[metadata_id] = record
        metadata_split_by_id[metadata_id] = split

print("Training annotations:", len(train_annotations))
print("Unique metadata records:", len(visualnews_by_id))

Training annotations: 71072
Unique metadata records: 466748


## Query Selection

Select 25 rows per class and construction category. Pristine queries are sampled independently to avoid repeating submitted captions.

In [3]:
selection_groups = [
    (False, 0), (False, 1), (False, 2), (False, 3),
    (True, 0), (True, 1), (True, 2), (True, 3),
]
selected_annotations = []
seen_captions = set()
seen_image_paths = set()
seen_pairs = set()
selection_rows = []

In [4]:
for falsified, source_dataset in selection_groups:
    class_candidates = train_annotations[train_annotations["falsified"] == falsified]
    candidates = class_candidates[class_candidates["source_dataset"] == source_dataset]
    candidates = candidates.sort_values(["id", "image_id"], kind="stable")
    candidates = candidates.sample(frac=1, random_state=seed)
    selected_count = 0
    rejected_count = 0

    for annotation in candidates.itertuples(index=False):
        caption_record = visualnews_by_id.get(annotation.id)
        image_record = visualnews_by_id.get(annotation.image_id)
        if caption_record is None or image_record is None:
            raise ValueError("A candidate query has an unresolved metadata ID")
        caption = caption_record["caption"]
        image_path = image_record["image_path"]
        pair = (annotation.id, annotation.image_id)

        if caption in seen_captions or image_path in seen_image_paths or pair in seen_pairs:
            rejected_count += 1
            continue

        selected_annotations.append(annotation)
        seen_captions.add(caption)
        seen_image_paths.add(image_path)
        seen_pairs.add(pair)
        selected_count += 1
        if selected_count == queries_per_group:
            break

    selection_rows.append({
        "falsified": falsified, "source_dataset": source_dataset,
        "selected": selected_count, "rejected_duplicate_candidates": rejected_count,
    })
    if selected_count != queries_per_group:
        raise ValueError(f"Insufficient unique queries for group {falsified}, {source_dataset}")

In [5]:
selected_queries = []
for annotation in selected_annotations:
    caption_record = visualnews_by_id[annotation.id]
    image_record = visualnews_by_id[annotation.image_id]
    selected_queries.append({
        "newsclip_split": "train",
        "id": annotation.id,
        "image_id": annotation.image_id,
        "falsified": annotation.falsified,
        "source_dataset": annotation.source_dataset,
        "submitted_caption": caption_record["caption"],
        "query_image_original_caption": image_record["caption"],
        "query_image_path": image_record["image_path"],
        "caption_source": caption_record["source"],
        "image_source": image_record["source"],
    })

In [6]:
query_table = pd.DataFrame(selected_queries)
query_ids = []
for number in range(1, len(query_table) + 1):
    query_ids.append(f"Q{number:04d}")
query_table.insert(0, "query_id", query_ids)

display(pd.DataFrame(selection_rows))
query_distribution = query_table.groupby(["falsified", "source_dataset"]).size()
display(query_distribution.rename("queries").reset_index())
with pd.option_context("display.max_colwidth", 90):
    display(query_table.head(5))

,falsified,source_dataset,selected,rejected_duplicate_candidates
0,False,0,25,0
1,False,1,25,0
2,False,2,25,0
3,False,3,25,0
4,True,0,25,25
5,True,1,25,25
6,True,2,25,25
7,True,3,25,25


,falsified,source_dataset,queries
0,False,0,25
1,False,1,25
2,False,2,25
3,False,3,25
4,True,0,25
5,True,1,25
6,True,2,25
7,True,3,25


,query_id,newsclip_split,id,image_id,falsified,source_dataset,submitted_caption,query_image_original_caption,query_image_path,caption_source,image_source
0,Q0001,train,720001,720001,False,0,Sense of panic Chris Evans and Alex Jones present the One Show from backstage,Sense of panic Chris Evans and Alex Jones present the One Show from backstage,visual_news/origin/guardian/images/0544/740.jpg,guardian,guardian
1,Q0002,train,123298,123298,False,0,Detail of an early20th century kitchen stove on display at the Tri River Pioneer Museu...,Detail of an early20th century kitchen stove on display at the Tri River Pioneer Museu...,visual_news/origin/washington_post/images/0102/650.jpg,washington_post,washington_post
2,Q0003,train,569824,569824,False,0,The Queen talks with the Australian high commissioner Alexander Downer as she prepares...,The Queen talks with the Australian high commissioner Alexander Downer as she prepares...,visual_news/origin/guardian/images/0294/375.jpg,guardian,guardian
3,Q0004,train,409076,409076,False,0,30 San Antonio Spurs Kyle Anderson SF,30 San Antonio Spurs Kyle Anderson SF,visual_news/origin/usa_today/images/0668/133.jpg,usa_today,usa_today
4,Q0005,train,525545,525545,False,0,Sierra Leone s President Ernest Bai Koroma left and Vice President Samuel SamSumana in...,Sierra Leone s President Ernest Bai Koroma left and Vice President Samuel SamSumana in...,visual_news/origin/guardian/images/0239/478.jpg,guardian,guardian


## Query Checks

IDs and construction categories are bookkeeping, not predictive model features.

In [7]:
def has_text(value):
    return type(value) is str and bool(value.strip())

caption_ids = set(query_table["id"])
image_ids = set(query_table["image_id"])
metadata_ids = set(visualnews_by_id)
missing_caption_ids = caption_ids - metadata_ids
missing_image_ids = image_ids - metadata_ids

pristine_queries = query_table[query_table["falsified"] == False]
falsified_queries = query_table[query_table["falsified"] == True]
falsified_counts = falsified_queries["source_dataset"].value_counts()
falsified_counts = falsified_counts.reindex([0, 1, 2, 3], fill_value=0)

query_content_columns = query_table.columns.drop("query_id")
query_duplicate_rows = int(query_table.duplicated(subset=query_content_columns).sum())
query_duplicate_pairs = int(query_table.duplicated(subset=["id", "image_id"]).sum())
query_duplicate_images = int(query_table["query_image_path"].duplicated().sum())
query_duplicate_captions = int(query_table["submitted_caption"].duplicated().sum())
missing_captions = int((~query_table["submitted_caption"].map(has_text)).sum())
missing_image_paths = int((~query_table["query_image_path"].map(has_text)).sum())

display(pd.DataFrame([{
    "duplicate_rows": query_duplicate_rows, "duplicate_pairs": query_duplicate_pairs,
    "duplicate_images": query_duplicate_images, "duplicate_captions": query_duplicate_captions,
    "missing_captions": missing_captions, "missing_image_paths": missing_image_paths,
    "unresolved_caption_ids": len(missing_caption_ids), "unresolved_image_ids": len(missing_image_ids),
}]))

,duplicate_rows,duplicate_pairs,duplicate_images,duplicate_captions,missing_captions,missing_image_paths,unresolved_caption_ids,unresolved_image_ids
0,0,0,0,0,0,0,0,0


In [8]:
query_checks = {
    "exactly 200 queries": len(query_table) == query_target,
    "100 pristine": len(pristine_queries) == 100,
    "100 falsified": len(falsified_queries) == 100,
    "25 falsified per category": bool((falsified_counts == 25).all()),
    "training queries only": bool((query_table["newsclip_split"] == "train").all()),
    "all caption IDs resolve": not missing_caption_ids,
    "all image IDs resolve": not missing_image_ids,
    "captions populated": missing_captions == 0,
    "image paths populated": missing_image_paths == 0,
    "unique query IDs": query_table["query_id"].is_unique,
    "no duplicate query rows": query_duplicate_rows == 0,
    "no duplicate query pairs": query_duplicate_pairs == 0,
    "no duplicate query images": query_duplicate_images == 0,
    "no duplicate submitted captions": query_duplicate_captions == 0,
}
display(pd.Series(query_checks, name="passed").to_frame())
if not all(query_checks.values()):
    raise ValueError("Query checks failed. Stop before freezing the pilot.")

,passed
exactly 200 queries,True
100 pristine,True
100 falsified,True
25 falsified per category,True
training queries only,True
all caption IDs resolve,True
all image IDs resolve,True
captions populated,True
image paths populated,True
unique query IDs,True


## Provenance Records

In [9]:
required_ids = caption_ids | image_ids
overlapping_ids = caption_ids & image_ids
provenance_records = []
for metadata_id in sorted(required_ids):
    provenance_records.append(visualnews_by_id[metadata_id])
provenance_table = pd.DataFrame(provenance_records)

provenance_summary = pd.DataFrame([{
    "caption_side_ids": len(caption_ids), "image_side_ids": len(image_ids),
    "overlapping_ids": len(overlapping_ids), "required_provenance_ids": len(required_ids),
}])
display(provenance_summary)
provenance_by_publisher = provenance_table["source"].value_counts()
display(provenance_by_publisher.rename("provenance_records").to_frame())

,caption_side_ids,image_side_ids,overlapping_ids,required_provenance_ids
0,200,200,100,300


,provenance_records
source,
guardian,129
usa_today,72
bbc,57
washington_post,42


## Evidence Pool

This pilot corpus guarantees provenance coverage and must not be treated as an independently sampled retrieval benchmark.

In [10]:
provenance_image_paths = set(provenance_table["image_path"])
query_image_paths = set(query_table["query_image_path"])
excluded_captions = set(query_table["submitted_caption"])
excluded_captions.update(query_table["query_image_original_caption"])

background_candidates = []
background_exclusions = Counter()
for record in metadata_splits["train"].values():
    metadata_id = record["id"]
    image_path = record["image_path"]
    caption = record["caption"]
    if metadata_id in required_ids:
        background_exclusions["required provenance ID"] += 1
        continue
    if image_path in provenance_image_paths:
        background_exclusions["provenance image path"] += 1
        continue
    if caption in excluded_captions:
        background_exclusions["submitted or image-original caption"] += 1
        continue
    background_candidates.append({"id": metadata_id, "source": record["source"], "image_path": image_path})

background_pool = pd.DataFrame(background_candidates)
background_pool = background_pool.sort_values("id", kind="stable")
duplicate_pool_paths = int(background_pool["image_path"].duplicated().sum())
if duplicate_pool_paths:
    background_pool = background_pool.drop_duplicates(subset=["image_path"], keep="first")

print("Eligible training background records:", len(background_pool))
print("Duplicate background paths excluded:", duplicate_pool_paths)
display(pd.Series(background_exclusions, name="excluded_records").to_frame())

Eligible training background records: 384703
Duplicate background paths excluded: 0


,excluded_records
required provenance ID,300


In [11]:
publishers = sorted(background_pool["source"].unique())
base_target = evidence_target // len(publishers)
extra_slots = evidence_target % len(publishers)
background_samples = []
publisher_targets = []

for publisher_index, publisher in enumerate(publishers):
    publisher_target = base_target
    if publisher_index < extra_slots:
        publisher_target += 1
    provenance_count = int(provenance_by_publisher.get(publisher, 0))
    background_needed = publisher_target - provenance_count
    if background_needed < 0:
        raise ValueError(f"Provenance exceeds the evidence target for {publisher}")

    publisher_candidates = background_pool[background_pool["source"] == publisher]
    if len(publisher_candidates) < background_needed:
        raise ValueError(f"Insufficient training background for {publisher}")
    sample = publisher_candidates.sample(n=background_needed, random_state=seed)
    background_samples.append(sample)
    publisher_targets.append({
        "source": publisher, "provenance": provenance_count,
        "background": background_needed, "final_target": publisher_target,
    })

selected_background = pd.concat(background_samples, ignore_index=True)
display(pd.DataFrame(publisher_targets))

,source,provenance,background,final_target
0,bbc,57,1193,1250
1,guardian,129,1121,1250
2,usa_today,72,1178,1250
3,washington_post,42,1208,1250


In [12]:
evidence_fields = [
    "id", "caption", "image_path", "article_path", "full_article_path",
    "source", "timestamp", "topic", "title",
]
evidence_records = list(provenance_records)
for metadata_id in selected_background["id"]:
    evidence_records.append(visualnews_by_id[metadata_id])

evidence_table = pd.DataFrame(evidence_records)
evidence_table = evidence_table[evidence_fields].copy()
evidence_table["is_query_provenance"] = evidence_table["id"].isin(required_ids)

provenance_roles = []
for metadata_id in evidence_table["id"]:
    if metadata_id in caption_ids and metadata_id in image_ids:
        role = "both"
    elif metadata_id in caption_ids:
        role = "caption"
    elif metadata_id in image_ids:
        role = "image"
    else:
        role = "background"
    provenance_roles.append(role)
evidence_table["provenance_role"] = provenance_roles

evidence_table = evidence_table.sort_values("id", kind="stable")
evidence_table = evidence_table.reset_index(drop=True)
background_table = evidence_table[evidence_table["is_query_provenance"] == False]
print("Evidence records:", len(evidence_table))
print("Background records:", len(background_table))

Evidence records: 5000
Background records: 4700


## Coverage

In [13]:
evidence_ids = set(evidence_table["id"])
covered_provenance_ids = required_ids & evidence_ids
missing_provenance_ids = required_ids - evidence_ids
provenance_coverage = 100 * len(covered_provenance_ids) / len(required_ids)

background_ids = set(background_table["id"])
background_paths = set(background_table["image_path"])
background_captions = set(background_table["caption"])
background_id_overlap = background_ids & required_ids
background_path_overlap = background_paths & provenance_image_paths
background_caption_overlap = background_captions & excluded_captions

required_article_paths = set(provenance_table["full_article_path"])
shared_article_rows = background_table["full_article_path"].isin(required_article_paths)
background_article_overlap = int(shared_article_rows.sum())

display(pd.DataFrame([{
    "required_provenance": len(required_ids), "covered_provenance": len(covered_provenance_ids),
    "coverage_pct": provenance_coverage, "missing_provenance": len(missing_provenance_ids),
    "background_ID_overlap": len(background_id_overlap),
    "background_image_overlap": len(background_path_overlap),
    "background_exact_caption_overlap": len(background_caption_overlap),
    "background_shared_full_article_rows": background_article_overlap,
}]))

,required_provenance,covered_provenance,coverage_pct,missing_provenance,background_ID_overlap,background_image_overlap,background_exact_caption_overlap,background_shared_full_article_rows
0,300,300,100.0,0,0,0,0,42


In [14]:
evidence_checks = {
    "exactly 5000 evidence records": len(evidence_table) == evidence_target,
    "unique evidence IDs": evidence_table["id"].is_unique,
    "unique evidence image paths": evidence_table["image_path"].is_unique,
    "all required provenance included": not missing_provenance_ids,
    "provenance flags consistent": int(evidence_table["is_query_provenance"].sum()) == len(required_ids),
    "no background provenance IDs": not background_id_overlap,
    "no background provenance images": not background_path_overlap,
    "no background query captions": not background_caption_overlap,
}
for field in ("caption", "image_path", "article_path", "full_article_path", "source", "timestamp", "topic"):
    populated = evidence_table[field].map(has_text)
    evidence_checks[f"{field} populated"] = bool(populated.all())

background_origins = selected_background["id"].map(metadata_split_by_id)
evidence_checks["training background only"] = bool((background_origins == "train").all())
display(pd.Series(evidence_checks, name="passed").to_frame())
if not all(evidence_checks.values()):
    raise ValueError("Evidence checks failed. Stop before freezing the pilot.")

,passed
exactly 5000 evidence records,True
unique evidence IDs,True
unique evidence image paths,True
all required provenance included,True
provenance flags consistent,True
no background provenance IDs,True
no background provenance images,True
no background query captions,True
caption populated,True
image_path populated,True


## Composition

In [15]:
publisher_composition = evidence_table.groupby(["source", "is_query_provenance"]).size()
publisher_composition = publisher_composition.unstack(fill_value=0)
publisher_composition = publisher_composition.rename(columns={False: "background", True: "provenance"})
publisher_composition["total"] = publisher_composition.sum(axis=1)
display(publisher_composition)

metadata_origins = evidence_table["id"].map(metadata_split_by_id)
display(metadata_origins.value_counts().rename("evidence_records").to_frame())

is_query_provenance,background,provenance,total
source,,,
bbc,1193,57,1250
guardian,1121,129,1250
usa_today,1178,72,1250
washington_post,1208,42,1250


,evidence_records
id,
train,5000


In [16]:
topic_counts = evidence_table["topic"].value_counts()
print("Unique evidence topics:", len(topic_counts))
display(topic_counts.rename("records").to_frame())

composition_checks = pd.DataFrame([{
    "duplicate_evidence_captions": int(evidence_table["caption"].duplicated().sum()),
    "duplicate_full_article_paths": int(evidence_table["full_article_path"].duplicated().sum()),
    "null_titles": int(evidence_table["title"].isna().sum()),
}])
display(composition_checks)

Unique evidence topics: 54


,records
topic,
arts_culture,1412
law_crime,515
international_relations,511
science_technology,377
politics_elections,282
sports,219
world,189
football,148
sport,144


,duplicate_evidence_captions,duplicate_full_article_paths,null_titles
0,0,266,1


In [17]:
reproducibility_info = {
    "seed": seed, "query_count": len(query_table),
    "pristine_count": len(pristine_queries), "falsified_count": len(falsified_queries),
    "evidence_target": evidence_target, "evidence_count": len(evidence_table),
    "provenance_count": len(required_ids), "background_count": len(background_table),
    "provenance_coverage_pct": provenance_coverage, "pandas_version": pd.__version__,
}
display(pd.Series(reproducibility_info, name="value").to_frame())
display(falsified_counts.rename("falsified_queries").to_frame())

,value
seed,42
query_count,200
pristine_count,100
falsified_count,100
evidence_target,5000
evidence_count,5000
provenance_count,300
background_count,4700
provenance_coverage_pct,100.0
pandas_version,3.0.6


,falsified_queries
source_dataset,
0,25
1,25
2,25
3,25


## Freeze Pilot

Existing frozen files must match this selection; differing files are preserved and raise an error.

In [18]:
parquet_available = importlib.util.find_spec("pyarrow") is not None
manifest_format = "parquet" if parquet_available else "csv"
query_manifest_path = manifest_directory / f"pilot_queries.{manifest_format}"
evidence_manifest_path = manifest_directory / f"pilot_evidence.{manifest_format}"


def read_manifest(path):
    if path.suffix == ".parquet":
        return pd.read_parquet(path, engine="pyarrow")
    return pd.read_csv(path, keep_default_na=False, na_values={"title": [""]})

print("Manifest format:", manifest_format)

Manifest format: parquet


In [19]:
manifest_directory.mkdir(parents=True, exist_ok=True)
query_file_exists = query_manifest_path.exists()
evidence_file_exists = evidence_manifest_path.exists()

if query_file_exists or evidence_file_exists:
    if not (query_file_exists and evidence_file_exists):
        raise ValueError("Only one frozen manifest exists. Review before writing.")
    existing_queries = read_manifest(query_manifest_path)
    existing_evidence = read_manifest(evidence_manifest_path)
    pd.testing.assert_frame_equal(query_table, existing_queries, check_dtype=False)
    pd.testing.assert_frame_equal(evidence_table, existing_evidence, check_dtype=False)
    print("Existing frozen manifests match; left unchanged.")
else:
    if manifest_format == "parquet":
        query_table.to_parquet(query_manifest_path, engine="pyarrow", index=False)
        evidence_table.to_parquet(evidence_manifest_path, engine="pyarrow", index=False)
    else:
        query_table.to_csv(query_manifest_path, index=False)
        evidence_table.to_csv(evidence_manifest_path, index=False)
    print("Created both pilot manifests.")

Existing frozen manifests match; left unchanged.


In [20]:
if not query_manifest_path.is_file() or not evidence_manifest_path.is_file():
    raise ValueError("Pilot manifest files are missing")

reloaded_queries = read_manifest(query_manifest_path)
reloaded_evidence = read_manifest(evidence_manifest_path)
required_query_columns = set(query_table.columns)
required_evidence_columns = set(evidence_table.columns)
reloaded_caption_ids = set(reloaded_queries["id"])
reloaded_image_ids = set(reloaded_queries["image_id"])
reloaded_required_ids = reloaded_caption_ids | reloaded_image_ids
reloaded_evidence_ids = set(reloaded_evidence["id"])

reload_checks = {
    "query row count": len(reloaded_queries) == query_target,
    "evidence row count": len(reloaded_evidence) == evidence_target,
    "query columns present": required_query_columns.issubset(reloaded_queries.columns),
    "evidence columns present": required_evidence_columns.issubset(reloaded_evidence.columns),
    "unique evidence IDs": reloaded_evidence["id"].is_unique,
    "all query provenance present": reloaded_required_ids.issubset(reloaded_evidence_ids),
}
pd.testing.assert_frame_equal(query_table, reloaded_queries, check_dtype=False)
pd.testing.assert_frame_equal(evidence_table, reloaded_evidence, check_dtype=False)
display(pd.Series(reload_checks, name="passed").to_frame())
if not all(reload_checks.values()):
    raise ValueError("Frozen manifest integrity checks failed")

,passed
query row count,True
evidence row count,True
query columns present,True
evidence columns present,True
unique evidence IDs,True
all query provenance present,True


In [21]:
file_rows = []
for manifest_path, records in (
    (query_manifest_path, reloaded_queries),
    (evidence_manifest_path, reloaded_evidence),
):
    file_rows.append({
        "path": str(manifest_path.relative_to(project_root)),
        "exists": manifest_path.is_file(), "rows": len(records),
        "size_kb": round(manifest_path.stat().st_size / 1_000, 2),
    })
display(pd.DataFrame(file_rows))

,path,exists,rows,size_kb
0,data/manifests/pilot_queries.parquet,True,200,45.96
1,data/manifests/pilot_evidence.parquet,True,5000,993.19


## Notes

- Pilot query set frozen with seed 42.
- Evidence corpus includes all required provenance records.
- This is a debugging corpus, not an unbiased retrieval benchmark.
- No images have been staged yet.
- Retrieval has not started.